In [ ]:
# RQ2: Impact of Data Augmentation on CNN Performance and Generalization Ability

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam


# 1. Create output folders

OUTPUT_DIR = Path("Results_RQ_Outputs")
BASE_DIR = OUTPUT_DIR / "RQ2_Data_Augmentation_Results"
FIG_DIR = BASE_DIR / "figures"
TABLE_DIR = BASE_DIR / "tables"
TABLE_PNG_DIR = BASE_DIR / "tables_png"
MODEL_DIR = BASE_DIR / "models"

for folder in [OUTPUT_DIR, BASE_DIR, FIG_DIR, TABLE_DIR, TABLE_PNG_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


# 2. Load dataset

styles = pd.read_csv("styles.csv", on_bad_lines="skip")
images = pd.read_csv("images.csv")

styles["id"] = styles["id"].astype(str)
styles["filename"] = styles["id"] + ".jpg"

if "filename" in images.columns:
    df = pd.merge(styles, images, on="filename", how="inner")
else:
    df = styles.copy()


# 3. Check image folder

IMAGE_FOLDER = Path("images")

df["image_path"] = df["filename"].apply(lambda x: str(IMAGE_FOLDER / x))
df = df[df["image_path"].apply(os.path.exists)].copy()

if len(df) == 0:
    raise ValueError("No images found. Check that JPG files are inside the images folder.")


# 4. Select top 5 article categories

df = df.dropna(subset=["articleType"])

top_classes = df["articleType"].value_counts().head(5).index
df = df[df["articleType"].isin(top_classes)].copy()

print("Selected classes:")
print(df["articleType"].value_counts())


# 5. Dataset description table

dataset_table = pd.DataFrame({
    "Description": [
        "Research Question",
        "Total images used",
        "Target column",
        "Selected classes",
        "Input image size",
        "Models compared"
    ],
    "Value": [
        "What is the impact of data augmentation techniques on CNN performance and generalization ability?",
        len(df),
        "articleType",
        ", ".join(top_classes),
        "128 x 128",
        "CNN without augmentation vs CNN with augmentation"
    ]
})

dataset_table.to_csv(TABLE_DIR / "rq2_dataset_description.csv", index=False)

plt.figure(figsize=(13, 4))
plt.axis("off")
plt.table(
    cellText=dataset_table.values,
    colLabels=dataset_table.columns,
    cellLoc="center",
    loc="center"
)
plt.title("RQ2 Dataset Description Table")
plt.tight_layout()
plt.savefig(TABLE_PNG_DIR / "rq2_dataset_description.png", dpi=300)
plt.show()


# 6. Class distribution

class_counts = df["articleType"].value_counts()

class_distribution = class_counts.reset_index()
class_distribution.columns = ["Article Type", "Number of Images"]
class_distribution.to_csv(TABLE_DIR / "rq2_class_distribution.csv", index=False)

plt.figure(figsize=(8, 5))
class_counts.plot(kind="bar")
plt.title("RQ2 Class Distribution")
plt.xlabel("Article Type")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "rq2_class_distribution.png", dpi=300)
plt.show()

plt.figure(figsize=(8, 3))
plt.axis("off")
plt.table(
    cellText=class_distribution.values,
    colLabels=class_distribution.columns,
    cellLoc="center",
    loc="center"
)
plt.title("RQ2 Class Distribution Table")
plt.tight_layout()
plt.savefig(TABLE_PNG_DIR / "rq2_class_distribution_table.png", dpi=300)
plt.show()


# 7. Train validation test split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["articleType"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["articleType"],
    random_state=42
)

split_table = pd.DataFrame({
    "Split": ["Training", "Validation", "Testing"],
    "Number of Images": [len(train_df), len(val_df), len(test_df)],
    "Percentage": [
        round(len(train_df) / len(df) * 100, 2),
        round(len(val_df) / len(df) * 100, 2),
        round(len(test_df) / len(df) * 100, 2)
    ]
})

split_table.to_csv(TABLE_DIR / "rq2_data_split.csv", index=False)

plt.figure(figsize=(7, 3))
plt.axis("off")
plt.table(
    cellText=split_table.values,
    colLabels=split_table.columns,
    cellLoc="center",
    loc="center"
)
plt.title("RQ2 Data Split Table")
plt.tight_layout()
plt.savefig(TABLE_PNG_DIR / "rq2_data_split.png", dpi=300)
plt.show()


# 8. Image generators

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

no_aug_train_datagen = ImageDataGenerator(rescale=1./255)

aug_train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator_no_aug = no_aug_train_datagen.flow_from_dataframe(
    train_df,
    x_col="image_path",
    y_col="articleType",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

train_generator_aug = aug_train_datagen.flow_from_dataframe(
    train_df,
    x_col="image_path",
    y_col="articleType",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = test_datagen.flow_from_dataframe(
    val_df,
    x_col="image_path",
    y_col="articleType",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(
    test_df,
    x_col="image_path",
    y_col="articleType",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

class_names = list(train_generator_no_aug.class_indices.keys())
num_classes = len(class_names)


# 9. Save augmentation setting table

augmentation_table = pd.DataFrame({
    "Model": ["CNN Without Augmentation", "CNN With Augmentation"],
    "Rescale": ["Yes", "Yes"],
    "Rotation": ["No", "20 degrees"],
    "Zoom": ["No", "0.2"],
    "Width Shift": ["No", "0.1"],
    "Height Shift": ["No", "0.1"],
    "Horizontal Flip": ["No", "Yes"]
})

augmentation_table.to_csv(TABLE_DIR / "rq2_augmentation_settings.csv", index=False)

plt.figure(figsize=(12, 3))
plt.axis("off")
plt.table(
    cellText=augmentation_table.values,
    colLabels=augmentation_table.columns,
    cellLoc="center",
    loc="center"
)
plt.title("RQ2 Data Augmentation Settings")
plt.tight_layout()
plt.savefig(TABLE_PNG_DIR / "rq2_augmentation_settings.png", dpi=300)
plt.show()


# 10. CNN model function

def build_cnn_model(num_classes):
    model = Sequential([
        Conv2D(32, (3, 3), activation="relu", input_shape=(128, 128, 3)),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Conv2D(64, (3, 3), activation="relu"),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Conv2D(128, (3, 3), activation="relu"),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.5),
        Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# 11. Train CNN without augmentation

cnn_no_aug = build_cnn_model(num_classes)

history_no_aug = cnn_no_aug.fit(
    train_generator_no_aug,
    validation_data=val_generator,
    epochs=10
)

cnn_no_aug.save(MODEL_DIR / "rq2_cnn_without_augmentation.h5")


# 12. Train CNN with augmentation

cnn_aug = build_cnn_model(num_classes)

history_aug = cnn_aug.fit(
    train_generator_aug,
    validation_data=val_generator,
    epochs=10
)

cnn_aug.save(MODEL_DIR / "rq2_cnn_with_augmentation.h5")


# 13. Evaluate both models

test_generator.reset()
pred_probs_no_aug = cnn_no_aug.predict(test_generator)
pred_classes_no_aug = np.argmax(pred_probs_no_aug, axis=1)
true_classes = test_generator.classes

test_accuracy_no_aug = accuracy_score(true_classes, pred_classes_no_aug)

test_generator.reset()
pred_probs_aug = cnn_aug.predict(test_generator)
pred_classes_aug = np.argmax(pred_probs_aug, axis=1)

test_accuracy_aug = accuracy_score(true_classes, pred_classes_aug)

print("Test Accuracy Without Augmentation:", test_accuracy_no_aug)
print("Test Accuracy With Augmentation:", test_accuracy_aug)


# 14. Save training accuracy comparison curve

plt.figure(figsize=(8, 5))
plt.plot(history_no_aug.history["accuracy"], label="Training Accuracy Without Augmentation")
plt.plot(history_aug.history["accuracy"], label="Training Accuracy With Augmentation")
plt.title("RQ2 Training Accuracy Comparison")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "rq2_training_accuracy_comparison.png", dpi=300)
plt.show()


# 15. Save validation accuracy comparison curve

plt.figure(figsize=(8, 5))
plt.plot(history_no_aug.history["val_accuracy"], label="Validation Accuracy Without Augmentation")
plt.plot(history_aug.history["val_accuracy"], label="Validation Accuracy With Augmentation")
plt.title("RQ2 Validation Accuracy Comparison")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "rq2_validation_accuracy_comparison.png", dpi=300)
plt.show()


# 16. Save training loss comparison curve

plt.figure(figsize=(8, 5))
plt.plot(history_no_aug.history["loss"], label="Training Loss Without Augmentation")
plt.plot(history_aug.history["loss"], label="Training Loss With Augmentation")
plt.title("RQ2 Training Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "rq2_training_loss_comparison.png", dpi=300)
plt.show()


# 17. Save validation loss comparison curve

plt.figure(figsize=(8, 5))
plt.plot(history_no_aug.history["val_loss"], label="Validation Loss Without Augmentation")
plt.plot(history_aug.history["val_loss"], label="Validation Loss With Augmentation")
plt.title("RQ2 Validation Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "rq2_validation_loss_comparison.png", dpi=300)
plt.show()


# 18. Save test accuracy bar chart

accuracy_values = [test_accuracy_no_aug, test_accuracy_aug]
accuracy_labels = ["Without Augmentation", "With Augmentation"]

plt.figure(figsize=(7, 5))
plt.bar(accuracy_labels, accuracy_values)
plt.title("RQ2 Test Accuracy Comparison")
plt.xlabel("Model")
plt.ylabel("Test Accuracy")
plt.ylim(0, 1)

for i, value in enumerate(accuracy_values):
    plt.text(i, value + 0.01, str(round(value * 100, 2)) + "%", ha="center")

plt.tight_layout()
plt.savefig(FIG_DIR / "rq2_test_accuracy_comparison.png", dpi=300)
plt.show()


# 19. Generalization gap table

final_train_acc_no_aug = history_no_aug.history["accuracy"][-1]
final_val_acc_no_aug = history_no_aug.history["val_accuracy"][-1]

final_train_acc_aug = history_aug.history["accuracy"][-1]
final_val_acc_aug = history_aug.history["val_accuracy"][-1]

generalization_gap_no_aug = final_train_acc_no_aug - final_val_acc_no_aug
generalization_gap_aug = final_train_acc_aug - final_val_acc_aug

comparison_table = pd.DataFrame({
    "Model": ["CNN Without Augmentation", "CNN With Augmentation"],
    "Final Training Accuracy": [
        round(final_train_acc_no_aug, 4),
        round(final_train_acc_aug, 4)
    ],
    "Final Validation Accuracy": [
        round(final_val_acc_no_aug, 4),
        round(final_val_acc_aug, 4)
    ],
    "Test Accuracy": [
        round(test_accuracy_no_aug, 4),
        round(test_accuracy_aug, 4)
    ],
    "Generalization Gap": [
        round(generalization_gap_no_aug, 4),
        round(generalization_gap_aug, 4)
    ],
    "Accuracy Percentage": [
        str(round(test_accuracy_no_aug * 100, 2)) + "%",
        str(round(test_accuracy_aug * 100, 2)) + "%"
    ]
})

comparison_table.to_csv(TABLE_DIR / "rq2_model_comparison.csv", index=False)

plt.figure(figsize=(14, 3))
plt.axis("off")
plt.table(
    cellText=comparison_table.values,
    colLabels=comparison_table.columns,
    cellLoc="center",
    loc="center"
)
plt.title("RQ2 Model Performance and Generalization Comparison")
plt.tight_layout()
plt.savefig(TABLE_PNG_DIR / "rq2_model_comparison.png", dpi=300)
plt.show()


# 20. Classification reports

report_no_aug = classification_report(
    true_classes,
    pred_classes_no_aug,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_aug = classification_report(
    true_classes,
    pred_classes_aug,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_no_aug_df = pd.DataFrame(report_no_aug).transpose()
report_aug_df = pd.DataFrame(report_aug).transpose()

report_no_aug_df.to_csv(TABLE_DIR / "rq2_classification_report_without_augmentation.csv")
report_aug_df.to_csv(TABLE_DIR / "rq2_classification_report_with_augmentation.csv")

plt.figure(figsize=(12, 6))
plt.axis("off")
plt.table(
    cellText=np.round(report_no_aug_df.values, 3),
    rowLabels=report_no_aug_df.index,
    colLabels=report_no_aug_df.columns,
    cellLoc="center",
    loc="center"
)
plt.title("RQ2 Classification Report Without Augmentation")
plt.tight_layout()
plt.savefig(TABLE_PNG_DIR / "rq2_classification_report_without_augmentation.png", dpi=300)
plt.show()

plt.figure(figsize=(12, 6))
plt.axis("off")
plt.table(
    cellText=np.round(report_aug_df.values, 3),
    rowLabels=report_aug_df.index,
    colLabels=report_aug_df.columns,
    cellLoc="center",
    loc="center"
)
plt.title("RQ2 Classification Report With Augmentation")
plt.tight_layout()
plt.savefig(TABLE_PNG_DIR / "rq2_classification_report_with_augmentation.png", dpi=300)
plt.show()


# 21. Confusion matrices

cm_no_aug = confusion_matrix(true_classes, pred_classes_no_aug)
cm_aug = confusion_matrix(true_classes, pred_classes_aug)

cm_no_aug_df = pd.DataFrame(cm_no_aug, index=class_names, columns=class_names)
cm_aug_df = pd.DataFrame(cm_aug, index=class_names, columns=class_names)

cm_no_aug_df.to_csv(TABLE_DIR / "rq2_confusion_matrix_without_augmentation.csv")
cm_aug_df.to_csv(TABLE_DIR / "rq2_confusion_matrix_with_augmentation.csv")

plt.figure(figsize=(8, 6))
plt.imshow(cm_no_aug)
plt.title("RQ2 Confusion Matrix Without Augmentation")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks(range(len(class_names)), class_names, rotation=45)
plt.yticks(range(len(class_names)), class_names)

for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(j, i, cm_no_aug[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.savefig(FIG_DIR / "rq2_confusion_matrix_without_augmentation.png", dpi=300)
plt.show()

plt.figure(figsize=(8, 6))
plt.imshow(cm_aug)
plt.title("RQ2 Confusion Matrix With Augmentation")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks(range(len(class_names)), class_names, rotation=45)
plt.yticks(range(len(class_names)), class_names)

for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(j, i, cm_aug[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.savefig(FIG_DIR / "rq2_confusion_matrix_with_augmentation.png", dpi=300)
plt.show()


# 22. Final interpretation table

if test_accuracy_aug > test_accuracy_no_aug:
    conclusion = "Data augmentation improved test accuracy and helped the CNN generalize better."
elif test_accuracy_aug < test_accuracy_no_aug:
    conclusion = "Data augmentation reduced test accuracy, possibly because the transformations were too strong or the model needed more training."
else:
    conclusion = "Data augmentation produced similar test accuracy compared with the non-augmented model."

interpretation_table = pd.DataFrame({
    "Research Question": [
        "What is the impact of data augmentation techniques on CNN performance and generalization ability?"
    ],
    "Main Finding": [conclusion],
    "Without Augmentation Accuracy": [str(round(test_accuracy_no_aug * 100, 2)) + "%"],
    "With Augmentation Accuracy": [str(round(test_accuracy_aug * 100, 2)) + "%"],
    "Generalization Meaning": [
        "A smaller gap between training and validation accuracy usually indicates better generalization."
    ]
})

interpretation_table.to_csv(TABLE_DIR / "rq2_final_interpretation.csv", index=False)

plt.figure(figsize=(15, 3))
plt.axis("off")
plt.table(
    cellText=interpretation_table.values,
    colLabels=interpretation_table.columns,
    cellLoc="center",
    loc="center"
)
plt.title("RQ2 Final Interpretation Table")
plt.tight_layout()
plt.savefig(TABLE_PNG_DIR / "rq2_final_interpretation.png", dpi=300)
plt.show()


# 23. Save all history values

history_table = pd.DataFrame({
    "Epoch": list(range(1, len(history_no_aug.history["accuracy"]) + 1)),
    "Train Accuracy Without Augmentation": history_no_aug.history["accuracy"],
    "Validation Accuracy Without Augmentation": history_no_aug.history["val_accuracy"],
    "Train Loss Without Augmentation": history_no_aug.history["loss"],
    "Validation Loss Without Augmentation": history_no_aug.history["val_loss"],
    "Train Accuracy With Augmentation": history_aug.history["accuracy"],
    "Validation Accuracy With Augmentation": history_aug.history["val_accuracy"],
    "Train Loss With Augmentation": history_aug.history["loss"],
    "Validation Loss With Augmentation": history_aug.history["val_loss"]
})

history_table.to_csv(TABLE_DIR / "rq2_training_history.csv", index=False)

print("RQ2 completed successfully.")
print("All outputs saved in:", BASE_DIR)
print("Figures saved in:", FIG_DIR)
print("CSV tables saved in:", TABLE_DIR)
print("PNG tables saved in:", TABLE_PNG_DIR)
print("Models saved in:", MODEL_DIR)